# NMODL to BrainCell Walkthrough

This notebook shows the inspectable conversion chain as currently implemented:

```text
.mod
-> parser AST
-> raw_blocks
-> canonical_blocks
-> bc_ast
-> semantic_ir
-> target_ir
-> rendered Python
-> render validation
```

The flow is no longer described as "CanonicalBlocks go straight into the template". The layering that actually matters now is:

- Step 1: `raw_blocks` / `canonical_blocks` / `bc_ast`
- Step 2: `semantic_ir` / `target_ir`
- Step 3: rendered Python / validation

This notebook uses three input files to illustrate the current behaviour:

- `kv.mod`: single-ion `inf/tau` positive case
- `na_alpha_beta.mod`: single-ion `alpha/beta` positive case
- `hh.mod`: negative case that builds in the semantic layer but is rejected by target lowering


In [ ]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "steps").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from steps import ast_to_json
from steps import resolve_mod_file
from steps import save_pipeline_artifacts
from steps import run_pipeline_from_path
from steps.inspect_ast import run as run_inspect_ast
from steps.inspect_ir import run as run_inspect_ir


def preview_json(value, limit=2000):
    text = json.dumps(value, indent=2, ensure_ascii=False)
    print(text[:limit])
    if len(text) > limit:
        print(f"\n... truncated, total_chars={len(text)}")


def summarize_step1(step1_result):
    return {
        "source_file": step1_result["source_file"],
        "ast_root_type": step1_result["ast_root_type"],
        "block_counts": step1_result["block_counts"],
        "nonempty_raw_blocks": [name for name, values in step1_result["raw_blocks"].items() if values],
        "bc_ast_keys": sorted(step1_result["bc_ast"].keys()),
    }


def summarize_step2(step2_result):
    return {
        "summary": step2_result["summary"],
        "semantic_ir_keys": sorted(step2_result["semantic_ir"].keys()),
        "target_ir_keys": sorted(step2_result["target_ir"].keys()),
    }


kv_mod = resolve_mod_file(str(repo_root / "mod_files" / "kv.mod"))
na_mod = resolve_mod_file(str(repo_root / "mod_files" / "na_alpha_beta.mod"))
hh_mod = resolve_mod_file(str(repo_root / "mod_files" / "hh.mod"))

step1_kv = run_inspect_ast(kv_mod)
step2_kv = run_inspect_ir(step1_kv)
pipeline_kv = run_pipeline_from_path(str(kv_mod))

print(kv_mod)


## 1. Starting from the input file

First look at this walkthrough's main positive case, `kv.mod`. It is currently the best mechanism for observing the full chain:

- a single `USEION`
- one ohmic current
- `inf/tau` gating dynamics


In [ ]:
print(step1_kv["reconstructed_nmodl"])


## 2. Parser AST summary

The raw parser AST is still the lowest-level source of truth, but the notebook no longer treats it as the only usable interface.
First look at the root node type and a JSON summary.


In [ ]:
ast_payload_kv = step1_kv["ast_json"]
print(step1_kv["ast_root_type"])
preview_json(ast_payload_kv, limit=1500)


## 3. Step 1: raw_blocks / canonical_blocks / bc_ast

Step 1 does more than "extract block text"; it produces three boundaries at once:

- `raw_blocks`: keeps the original block text as far as possible
- `canonical_blocks`: block-level canonicalization
- `bc_ast`: BrainCell's own typed AST


In [ ]:
preview_json(summarize_step1(step1_kv), limit=2000)


In [ ]:
print("NEURON canonical block:")
preview_json(step1_kv["canonical_blocks"]["NEURON"], limit=1200)

print("\nBREAKPOINT canonical block:")
preview_json(step1_kv["canonical_blocks"]["BREAKPOINT"], limit=1200)

print("\nbc_ast summary:")
preview_json({
    "mechanism_name": step1_kv["bc_ast"]["mechanism_name"],
    "useions": step1_kv["bc_ast"]["useions"],
    "states": step1_kv["bc_ast"]["states"],
    "solve_blocks": step1_kv["bc_ast"]["solve_blocks"],
}, limit=2000)


## 4. Step 2: semantic_ir

`semantic_ir` is the most important new boundary of the current implementation. It recovers:

- `symbols`, from the symbol-table point of view
- `procedure_env`, after procedure expansion
- the current expressions in `BREAKPOINT`
- `gate_kinetics`, after the state equations are recognized
- `unsupported_features`


In [ ]:
preview_json({
    "summary": step2_kv["summary"],
    "procedure_env": step2_kv["semantic_ir"]["procedure_env"],
    "currents": step2_kv["semantic_ir"]["currents"],
    "gate_kinetics": step2_kv["semantic_ir"]["gate_kinetics"],
    "unsupported_features": step2_kv["semantic_ir"]["unsupported_features"],
}, limit=3500)


## 5. Step 2: target_ir

`target_ir` is already the target layer the BrainCell renderer can consume directly.
For the current density-channel path, focus on:

- `target_family`
- `base_class_name`
- `root_type`
- `g_max_param`
- `gates`
- `current_model`
- `supported` / `rejection_reasons`


In [ ]:
preview_json({
    "summary": step2_kv["summary"],
    "g_max_param": step2_kv["target_ir"]["g_max_param"],
    "gates": step2_kv["target_ir"]["gates"],
    "current_model": step2_kv["target_ir"]["current_model"],
}, limit=4000)


## 6. Step 3: rendered Python and validation

Step 3 does more than render; it also validates the generated code for real:

- `compile`
- `exec`
- obtain the target class

This is closer to actual usability than the old "the template text can be printed".


In [ ]:
step3_kv = pipeline_kv["step3_result"]
preview_json(step3_kv["summary"], limit=2000)
print("\nRendered preview:\n")
print("\n".join(step3_kv["rendered_text"].splitlines()[:80]))


## 7. Saving the walkthrough artifacts

The next step saves the core artifacts of the current positive case to `examples/artifacts/kv/`:

- `ast.json`
- `raw_blocks.json`
- `canonical_blocks.json`
- `bc_ast.json`
- `semantic_ir.json`
- `target_ir.json`
- `rendered_channel.py`
- `validation.json`
- `metadata.json`


In [ ]:
artifact_dir = repo_root / "examples" / "artifacts" / kv_mod.stem
artifact_summary = save_pipeline_artifacts(pipeline_kv, artifact_dir)
preview_json(artifact_summary, limit=2000)


## 8. A second positive case: `alpha/beta`

`na_alpha_beta.mod` follows another target family:

- `target_family = hh_ohmic_alpha_beta`
- the semantic layer keeps `alpha_expression` / `beta_expression`
- the target layer derives an `inf/tau` view for rendering


In [ ]:
step1_na = run_inspect_ast(na_mod)
step2_na = run_inspect_ir(step1_na)
pipeline_na = run_pipeline_from_path(str(na_mod))

preview_json({
    "summary": step2_na["summary"],
    "gate_kinetics": step2_na["semantic_ir"]["gate_kinetics"],
    "target_family": step2_na["target_ir"]["target_family"],
    "validation": pipeline_na["step3_result"]["validation"],
}, limit=3500)


## 9. Negative case: why the current version rejects multi-ion mechanisms

`hh.mod` shows that the current rejection is neither a parser failure nor a semantic recovery failure.
What actually happens is:

- `semantic_ir` still builds
- but `target_ir` is rejected because of `multi_useion` and `nonspecific_current`
- so it should not go on to the current renderer


In [ ]:
step1_hh = run_inspect_ast(hh_mod)
step2_hh = run_inspect_ir(step1_hh)

preview_json({
    "summary": step2_hh["summary"],
    "semantic_useions": step2_hh["semantic_ir"]["useions"],
    "semantic_unsupported_features": step2_hh["semantic_ir"]["unsupported_features"],
    "target_supported": step2_hh["target_ir"]["supported"],
    "target_rejection_reasons": step2_hh["target_ir"]["rejection_reasons"],
}, limit=3000)


## 10. The corresponding CLI

Once the layer-by-layer view in the notebook is clear, the corresponding CLI commands are:

```bash
python examples/convert_mod/nmodl/steps/inspect_ast/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/steps/inspect_ir/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/steps/render/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/examples/generate_braincell.py   examples/convert_mod/nmodl/mod_files/kv.mod
```

The recommended order is still: look at `bc_ast` first, then `semantic_ir`, then `target_ir`, and only then care about rendering and validation.
